# T1：在 Google Colab 运行第4节离线评估

**操作：** 选择 CPU 运行时 → 运行全部 → 出现上传按钮时选择 `replication_package_MS-FTBDA-20-01585.zip` → 查看成本表 → 下载结果 ZIP。

本Notebook保留作者第4节代码的计算逻辑。评估数据已有模型预测，不执行神经网络训练。数据有3000行、2607个不同SKU标识，结果不要求等于论文完整样本。

文件上传发生在你的Colab会话中。`/content` 是临时存储；运行结束请下载结果，也可以使用最后的可选Google Drive备份单元。

## 1. 检查依赖（仅安装缺少的包）

In [ ]:
import importlib.util
import subprocess
import sys

packages = ['numpy', 'pandas', 'scipy', 'seaborn', 'tqdm', 'matplotlib', 'jinja2']
missing = [name for name in packages if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('依赖检查完成；本实验使用CPU即可。')

## 2. 上传原始压缩包并读取评估数据
点击下方运行后出现的文件选择按钮。保持ZIP原样上传，无需先解压。

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import os
import hashlib

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('请只上传一个 replication_package_MS-FTBDA-20-01585.zip 文件。')
zip_name = zip_names[0]
archive_bytes = uploaded[zip_name]
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
ROOT = Path('/content/t1_reproduction')
ROOT.mkdir(parents=True, exist_ok=True)
with ZipFile(BytesIO(archive_bytes)) as archive:
    candidates = [name for name in archive.namelist()
                  if Path(name).name == 'df_test.csv' and not name.startswith('__MACOSX/')]
    if len(candidates) != 1:
        raise ValueError('压缩包中应当恰有一个 df_test.csv。请核对上传文件。')
    csv_bytes = archive.read(candidates[0])
    (ROOT / 'df_test.csv').write_bytes(csv_bytes)
input_sha256 = hashlib.sha256(csv_bytes).hexdigest()
os.chdir(ROOT)
print('工作目录:', ROOT)
print('读取数据:', candidates[0])
print('数据SHA256:', input_sha256)


## 3. 作者第4节代码
以下计算单元与包中 `E2E_Section4-checkpoint.ipynb` 的非空代码单元一致。主要流程：解析数据 → 基准预测 → 按SKU聚合 → 库存轨迹 → 成本统计。

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
from tqdm import tqdm
import math
import seaborn as sns

from scipy.stats import gamma
import datetime as dt

In [ ]:
# df_test = pd.read_pickle("df_test.pkl")
df_test = pd.read_csv("df_test.csv")
len(df_test)

In [ ]:
#Correct data type
df_test['sf_rnn'] = df_test['sf_rnn'].apply(lambda x: x.strip('][').split(', '))
df_test['sf_rnn'] = df_test['sf_rnn'].apply(lambda x: pd.to_numeric(x))
df_test['demand_hist'] = df_test['demand_hist'].apply(lambda x: pd.to_numeric(x.strip('][').split()))

#### PTO benchmark1 and benchmark 2

In [ ]:
df_test['Bm1_pred'] = df_test['sf_rnn'].apply(lambda x: np.mean(x))
df_test['Bm1_pred'] = df_test['Bm1_pred'] * (df_test['review_period'] + df_test['vlt']).astype(int)

b = 9
h = 1
def get_bm2(x):
    rl = x['review_period'] + x['vlt']
    if rl <= b:
        days = int(rl)
    else:
        days = int(rl) - rl//(b+h)
    return x['Bm2_pred'] * days

df_test['Bm2_pred'] = df_test['sf_rnn'].apply(lambda x: np.mean(x))
df_test['Bm2_pred'] = df_test.apply(get_bm2, axis=1)

#### Normal benchmark

In [ ]:
Z90 = 1.64
df_test['Normal_pred'] = df_test.apply(lambda x: int(x['demand_mean']*(x['review_period']+x['vendor_vlt_mean'])
                                       +Z90*np.sqrt((x['review_period']+x['vendor_vlt_mean'])*x['demand_std']**2
                                                    + x['demand_std']**2 * x['vendor_vlt_std'])), axis=1)

#### Gamma benchmark

In [ ]:
def gamma_base(x):
    mean = x['demand_mean']
    var = x['demand_std']**2
    theta = var/(mean+1e-4)
    k = mean/(theta+1e-4)
    k_sum = int(x['review_period']+x['vendor_vlt_mean'])*k
    gamma_stock = gamma.ppf(0.9, a=k_sum, scale = theta)
    if(np.isnan(gamma_stock)):
        return 0
    else:
        return int(gamma_stock)
df_test['Gamma_pred'] = df_test.apply(gamma_base, axis=1)

In [ ]:
df_test = df_test.groupby('SKU').agg(lambda x: x.tolist())

### Sequential test model

In [ ]:
def get_agginv(x, name):
    inv1, inv2 = [x['initial_stock'][0]], []
    rd = len(x['OPT_pred'])
 
    for r in range(rd):
        if r < rd - 1:
            len_day = len(x['demand_hist'][r])-1
        else:
            len_day = len(x['demand_hist'][r])
        for t in range(len_day):
            if t == 0:
                if r == 0:
                    replen = int(round(x[name+'_pred'][r] - inv1[0]))
                else:
                    try:
                        replen = int(round(x[name+'_pred'][r] - inv1[-int(round(x['vlt'][r]))-1]))
                    except:
                        replen = int(round(x[name+'_pred'][r] - inv1[1]))
            if t < int(round(x['vlt'][r])):
                if r == 0:
                    inv1.append(inv1[-1] - x['demand_hist'][r][t])
            elif t == int(round(x['vlt'][r])):
                if inv1[-1] >= 0:
                    inv_ = inv1[-1] + replen - x['demand_hist'][r][t]
                else:
                    inv_ = replen - x['demand_hist'][r][t]
                inv1.append(inv_)
                inv2.append(inv_)
            else:
                inv_ = inv1[-1] - x['demand_hist'][r][t]
                inv1.append(inv_)
                inv2.append(inv_)
    
    inv1 = inv1[1:]
    return [inv1, inv2]

In [ ]:
df_test['OPT_agginv_f'], df_test['OPT_agginv'] = zip(*df_test.apply(get_agginv, name='OPT',  axis=1))
df_test['E2E_RNN_agginv_f'], df_test['E2E_RNN_agginv'] = zip(*df_test.apply(get_agginv, name='E2E_RNN',  axis=1))
df_test['Bm1_agginv_f'], df_test['Bm1_agginv'] = zip(*df_test.apply(get_agginv, name='Bm1',  axis=1))
df_test['Bm2_agginv_f'], df_test['Bm2_agginv'] = zip(*df_test.apply(get_agginv, name='Bm2',  axis=1))
df_test['Normal_agginv_f'], df_test['Normal_agginv'] = zip(*df_test.apply(get_agginv, name='Normal',  axis=1))
df_test['Gamma_agginv_f'], df_test['Gamma_agginv'] = zip(*df_test.apply(get_agginv, name='Gamma',  axis=1))
df_test['gbm_agginv_f'], df_test['gbm_agginv'] = zip(*df_test.apply(get_agginv, name='gbm',  axis=1))

### Calculate cost

In [ ]:
h = 1
b = 9
str_list = ['OPT', 'E2E_RNN', 'Bm1', 'Bm2', 'Normal', 'Gamma', 'gbm']
numberOfRows = len(df_test)
df_cost_agg = pd.DataFrame(index=np.arange(0, numberOfRows), columns=str_list)
df_holding_agg = pd.DataFrame(index=np.arange(0, numberOfRows), columns=str_list)
df_back_agg = pd.DataFrame(index=np.arange(0, numberOfRows), columns=str_list)
df_stockout_agg = pd.DataFrame(index=np.arange(0, numberOfRows), columns=str_list)
df_turnover_agg = pd.DataFrame(index=np.arange(0, numberOfRows), columns=str_list)


df_test_ = df_test.reset_index(drop=True)
for str1 in str_list:
    str2 = str1 + '_agginv'
    df_holding_agg[str1] = df_test_[str2].apply(lambda x: h * sum([inv for inv in x if inv>0]) )
    df_back_agg[str1] = df_test_[str2].apply(lambda x: b * -sum([inv for inv in x if inv<0]) )
    df_cost_agg[str1] = df_holding_agg[str1] + df_back_agg[str1] 

In [ ]:
df_aggcom = pd.DataFrame({'Total cost': df_cost_agg[str_list].mean(),
             'Holding cost': df_holding_agg[str_list].mean(),
             'Stockout cost': df_back_agg[str_list].mean()}).T
df_aggcom

In [ ]:
print(df_aggcom.to_latex(float_format=lambda x: '%.2f' % x))

## 4. 保存结果与环境信息

下表为本地运行相同作者代码获得的参照值。程序显示与本地参照值的差异；有差异时先检查输入和库版本，不要为追平数字改变算法。

这里的“OPT”沿用作者字段名称，其下界含义仍需要结合代码口径审计。

In [ ]:
import json
import platform
from importlib.metadata import version
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT = ROOT / 'outputs' / run_id
OUTPUT.mkdir(parents=True, exist_ok=True)
df_aggcom.to_csv(OUTPUT / 'section4_original_costs.csv', encoding='utf-8-sig')
versions = {'python': platform.python_version()}
for name in packages:
    versions[name] = version(name)
report = {
    'run_id_utc': run_id,
    'archive_name': zip_name,
    'archive_sha256': archive_sha256,
    'input_sha256': input_sha256,
    'input_rows': len(pd.read_csv(ROOT / 'df_test.csv')),
    'unique_sku_groups': len(df_test),
    'versions': versions,
    'scope': 'Author Section 4 evaluation on disclosed subset; no model training',
}
(OUTPUT / 'run_metadata.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
reference = pd.Series({'OPT':2572.96, 'E2E_RNN':3036.76, 'Bm1':3842.31,
                       'Bm2':3837.83, 'Normal':3961.69, 'Gamma':4099.95, 'gbm':3424.87})
comparison = pd.DataFrame({'Colab total cost':df_aggcom.loc['Total cost'],
                           'Local reference (rounded)':reference})
comparison['Difference'] = comparison.iloc[:,0] - comparison.iloc[:,1]
comparison.to_csv(OUTPUT / 'reference_comparison.csv', encoding='utf-8-sig')
print('输入行数:', report['input_rows'], 'SKU组数:', report['unique_sku_groups'])
display(df_aggcom.round(2))
display(comparison.round(4))
print('结果目录:', OUTPUT)


## 5. 下载结果
运行后浏览器会下载ZIP。如果浏览器拦截下载，可在左侧文件面板找到打印出的ZIP路径并手动下载。

In [ ]:
from zipfile import ZIP_DEFLATED
RESULT_ZIP = ROOT / f't1_section4_results_{run_id}.zip'
with ZipFile(RESULT_ZIP, 'w', compression=ZIP_DEFLATED) as archive:
    for result_file in OUTPUT.iterdir():
        if result_file.is_file():
            archive.write(result_file, arcname=result_file.name)
print('结果压缩包:', RESULT_ZIP)
files.download(str(RESULT_ZIP))


## 6. 可选：备份到自己的 Google Drive

默认关闭。需要备份时，将下面 `BACKUP_TO_DRIVE` 改为 `True`，运行此单元并完成Google授权。结果保存到 `MyDrive/T1_reproduction/results/`。

In [ ]:
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    drive_output = Path('/content/drive/MyDrive/T1_reproduction/results')
    drive_output.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RESULT_ZIP, drive_output / RESULT_ZIP.name)
    print('已备份到:', drive_output / RESULT_ZIP.name)
else:
    print('Drive备份未启用；请保留已下载的结果ZIP。')


## 下一阶段

先对一个SKU手算库存与成本，再复现实地实验统计。完整E2E训练需要额外训练代码和数据；当前压缩包只含少量原始数据示例。

待审计点：Normal基准的分位数及方差公式、BM1/BM2预测构造、预测值是否为目标库存、到货时负库存的处理。此Notebook未修改这些作者计算规则。

Colab官方说明：https://research.google.com/colaboratory/faq.html
